# fixed-income-lab: worked examples

A tour of the four V1 instrument types plus scenario analysis and portfolio roll-up.
Run from the repository root after `pip install -e ".[dev]"`.

In [ ]:
from datetime import date

from fixed_income import analyze_bond, analyze_cash_flows
from fixed_income.conventions import Actual360, Actual365Fixed, Frequency
from fixed_income.instruments.amortizing import AmortizingBond
from fixed_income.instruments.bond import Bond, ZeroCouponBond
from fixed_income.instruments.floating_rate import FloatingRateNote
from fixed_income.instruments.rate_index import RateIndex, RateObservation
from fixed_income.pricing import YieldConvention
from fixed_income.portfolio import PortfolioPosition, analyze_portfolio
from fixed_income.risk import run_rate_shock_scenarios

## 1. Fixed-rate bond: price, yield, accrued interest, risk

In [ ]:
bond = Bond(
    face_value=100.0,
    coupon_rate=0.05,
    issue_date=date(2020, 1, 15),
    maturity_date=date(2030, 1, 15),
    frequency=Frequency.SEMI_ANNUAL,
    day_count=Actual365Fixed(),
)
settlement = date(2024, 4, 15)

result = analyze_bond(bond, settlement, yield_to_maturity=0.045)
result

In [ ]:
# Round trip: solving from the clean price we just computed recovers the same yield.
round_trip = analyze_bond(bond, settlement, clean_price=result.clean_price)
round_trip.yield_to_maturity

## 2. Zero-coupon bond

In [ ]:
zcb = ZeroCouponBond(100.0, 0.0, date(2022, 1, 1), date(2032, 1, 1))
analyze_bond(zcb, settlement, yield_to_maturity=0.05)

## 3. Floating-rate note

The first two reset dates use observed fixings; every later reset falls back to
the explicit forward-rate assumption. `rate_provenance()` shows which is which.

In [ ]:
index = RateIndex(
    name="SOFR-3M",
    observations=(
        RateObservation(date(2023, 1, 15), 0.0530),
        RateObservation(date(2023, 4, 17), 0.0525),
    ),
    forward_assumption=0.045,
    floor=0.0,
)
frn = FloatingRateNote(
    face_value=100.0,
    spread=0.0025,
    issue_date=date(2023, 1, 15),
    maturity_date=date(2025, 1, 15),
    rate_index=index,
    frequency=Frequency.QUARTERLY,
    day_count=Actual360(),
)
frn.rate_provenance()

In [ ]:
frn_settlement = frn.issue_date
yc = YieldConvention.street(frn.frequency)
analyze_cash_flows(
    frn.cash_flows_after(frn_settlement), frn.face_value, frn_settlement, yc, frn.day_count,
    yield_to_maturity=0.0475,
)

## 4. Amortizing bond (level principal) and factor analytics

In [ ]:
amortizing = AmortizingBond.with_level_principal(
    original_face=1_000_000.0,
    coupon_rate=0.06,
    issue_date=date(2022, 1, 1),
    maturity_date=date(2027, 1, 1),
    frequency=Frequency.SEMI_ANNUAL,
)
for entry in amortizing.amortization_schedule():
    print(entry)

## 5. Rate-shock scenario analysis: full reprice vs. Taylor approximation

In [ ]:
yc = YieldConvention.street(bond.frequency)
cash_flows = bond.cash_flows_after(settlement)
scenarios = run_rate_shock_scenarios(cash_flows, bond.face_value, settlement, 0.045, yc, bond.day_count)
for s in scenarios:
    print(f"{s.shock_bp:+.0f}bp: full={s.full_reprice:.4f}  duration_approx={s.duration_approx_price:.4f}  convexity_approx={s.convexity_approx_price:.4f}")

## 6. Portfolio roll-up

In [ ]:
position_a = PortfolioPosition("BOND-A", par_amount=5_000_000.0, maturity_date=bond.maturity_date, analytics=result)
zcb_analytics = analyze_bond(zcb, settlement, yield_to_maturity=0.05)
position_b = PortfolioPosition("ZERO-B", par_amount=2_000_000.0, maturity_date=zcb.maturity_date, analytics=zcb_analytics)

analyze_portfolio([position_a, position_b])